# 11 — Official Deepnote Native 8-Task Benchmark: Zero-Contamination Multi-Seed & Frontier Evaluation

**Self-Contained Publication Suite (Resolving Review Issues `M2`, `M3`, `M7`, `M8`, `E1`–`E4`, `E8`, `Z2`, `Z5`, `Z6`)**:
1. **Programmatic Zero-Contamination Audit (`M2`)**: Verifies before running a single token that all task-specific `important_mistakes` hints have been purged from source prompts.
2. **5-Way Controlled Architectural Ablation (`E1`, `E8`)**:
   - `B0_official_direct`: Official Deepnote prompt (`assess_recent_slack.txt`).
   - **`B0_prompt_matched_direct` (`E1`)**: Single-agent Direct given the **exact same** `GENERAL_ENTERPRISE_DISCIPLINE_BLOCK` as our multi-agent system.
   - `DIAL_v1_flat`: Flat multi-channel snapshot (ablation of Per-Thread Isolation).
   - `DIAL_Collab_no_stage3_audit` (`E8`): Per-Thread Closed-Loop Gate + Skill Routing, **without** Stage-3 Staged Write Buffer & Pre-Delivery Skeptic-Auditor.
   - **`DIAL_Collab_full` (Ours)**: Per-Thread Closed-Loop Gate (`deliberative_collab`) + Deep Silent Recon + Skill Routing + **Stage-3 Staged Draft–Audit–Refine**.
3. **Multi-Seed (`3` seeds) + Frontier Backbones (`E2`, `E3`)**: Runs across `gpt-4.1-mini`, `gpt-4.1`, and frontier `gpt-5.6-terra` (or `gpt-6.1-terra`) over seeds `(42, 1042, 2042)` with atomic resume.
4. **Cross-Family Multi-Judge (`E4`) & Exact Sign-Flip Permutation Test (`Z2`)**: Grades with `gpt-4.1` + `DeepSeek-V4.1-Flash` averaged, reports Error Taxonomy (`Z5`) & uncached USD cost (`Z6`).

In [ ]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "datasets/pagemantle_hard",
    "caches",
    "runs/deepnote_official_8_v3_clean",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)

In [ ]:
# === CELL 1: Sync GitHub Repo (git clone / git pull), Install Dependencies & Hot-Reload ===
REPO_URL = os.environ.get("DIAL_SKILL_REPO_URL", "https://github.com/BoldHu/multiagent-speak.git")
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

import importlib, sys, types

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    try:
        import typesafe_sdk, duckdb, typer, tenacity, scipy, sklearn  # noqa: F401
        !pip install -q -e . --no-deps
    except ImportError:
        !pip install -q -e .

_src_dir = str((REPO_DIR / "src").resolve())
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")

In [ ]:
# === CELL 2: Bootstrap Multi-Provider Secrets & Configs (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import (
    describe_secret_sources,
    load_autodl_api_key,
    load_jev_api_key,
    load_openai_api_key,
)
from dial_skill.config import load_dial_skill_config, load_experiments_config, load_models_config
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role

print(
    "Secret lookup diagnostics:",
    json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT, include_autodl=True), indent=2),
)
OPENAI_API_KEY = load_openai_api_key(drive_root=DRIVE_ROOT, required=True)
JEV_API_KEY = load_jev_api_key(drive_root=DRIVE_ROOT, required=True)
AUTODL_API_KEY = load_autodl_api_key(drive_root=DRIVE_ROOT, required=False)
AUTODL_BASE_URL = os.environ.get("AUTODL_BASE_URL", "https://www.autodl.art/api/v1")

MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print(
    f"OpenAI key: {bool(OPENAI_API_KEY)} | Jev key: {bool(JEV_API_KEY)} | AutoDL key: {bool(AUTODL_API_KEY)}"
)

## Step 1 — Programmatic Zero-Contamination Check & Deepnote 11-Service Runtime Setup

In [ ]:
# === CELL 3: Zero-Contamination Verification + Official Deepnote Dataset Setup ===
import hashlib, shutil

# 1. Hard programmatic assertion: ZERO task-specific Official-8 hints in prompts!
BANNED_PHRASES = [
    "non-production tests from production recovery",
    "recurring calendar masters from single-date",
    "completed past patches from open defects",
    "event counts from distinct users",
    "recorded system owner + Security review",
]
for rel in ("src/dial_skill/benchmarks/deepnote_official.py", "src/dial_skill/prompts/templates.py"):
    src_txt = (REPO_DIR / rel).read_text(encoding="utf-8")
    for leak in BANNED_PHRASES:
        assert leak not in src_txt, f"CONTAMINATION DETECTED in {rel}: {leak}"
print("[AUDIT PASSED] Zero task-specific Official-8 prompt contamination in codebase!")

DEEPNOTE_REPO_URL = "https://github.com/deepnote/agent-initiative-benchmark.git"
DEEPNOTE_REPO_DIR = (REPO_DIR / "external" / "agent-initiative-benchmark").resolve()
DEEPNOTE_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (DEEPNOTE_REPO_DIR / "tasks" / "index.json").is_file():
    !git clone --depth 1 {DEEPNOTE_REPO_URL} {DEEPNOTE_REPO_DIR}

if IN_COLAB:
    !pip install -q fastembed hnswlib ijson pypdf
    !pip install -q --no-deps -e {DEEPNOTE_REPO_DIR}

_dn_src = str((DEEPNOTE_REPO_DIR / "src").resolve())
if _dn_src not in sys.path:
    sys.path.insert(0, _dn_src)

OFFICIAL_DATA_DIR = Path("/content/pagemantle_official") if IN_COLAB else (DRIVE_ROOT / "pagemantle_official")
OFFICIAL_CACHE_DIR = Path("/content/pagemantle_official_cache") if IN_COLAB else (DRIVE_ROOT / "pagemantle_official_cache")
OFFICIAL_WORK_ROOT = Path("/content/pagemantle_official_work") if IN_COLAB else (DRIVE_ROOT / "pagemantle_official_work")
OFFICIAL_WORK_ROOT.mkdir(parents=True, exist_ok=True)

from benchmark.dataset import Release, prepare_data

release = Release()
expected_sha = release.lock["dataset"]["manifest_sha256"]
drive_pm = DRIVE_ROOT / "datasets" / "pagemantle"
use_source = None
if (drive_pm / "manifest.json").is_file():
    actual_sha = hashlib.sha256((drive_pm / "manifest.json").read_bytes()).hexdigest()
    has_all = all((drive_pm / pkg["path"]).is_file() for pkg in release.packages.values())
    if actual_sha == expected_sha and has_all and (drive_pm / "file-index.jsonl.gz").is_file():
        use_source = drive_pm
        print("Reusing verified Deepnote archives from Google Drive (0 GB download)!")

prep_info = prepare_data(root=OFFICIAL_DATA_DIR, source=use_source, cache_dir=OFFICIAL_CACHE_DIR, release=release)
print("Official dataset ready:", prep_info)

## Step 2 — Configure Multi-Seed (`3` Seeds), Backbones (incl. Frontier) & Cross-Family Judge

In [ ]:
# === CELL 4: Configure Backbones, Seeds & Atomic Checkpointing ===
from dial_skill.benchmarks.deepnote_official import (
    DeepnoteOfficialJudge,
    DeepnoteOfficialTaskSession,
    DeepnoteRubricJudgment,
    OfficialDialSkillAgent,
    OfficialDialSkillV2Agent,
    OfficialDirectAgent,
    OfficialEpisodeResult,
    OfficialPromptMatchedDirectAgent,
    load_deepnote_official_catalog,
    summarize_official_8_results,
)
from dial_skill.core.pipeline import DialSkillAgent
from dial_skill.core.skill_store import SkillStore
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache

EVAL_MODELS = ["gpt-4.1-mini", "gpt-4.1", "gpt-5.6-terra"]  # Add "gpt-6.1-terra" if enabled on account
SEEDS = [42, 1042, 2042]  # Set to [42] for fast single-seed check, [42, 1042, 2042] for full paper table
PRIMARY_JUDGE_MODEL = "gpt-4.1"
SECONDARY_JUDGE_MODEL = "DeepSeek-V4.1-Flash" if AUTODL_API_KEY else None

RUN_DIR = DRIVE_ROOT / "runs" / "deepnote_official_8_v3_clean"
EPISODES_JSONL = RUN_DIR / "episodes_multiseed_v3.jsonl"
JUDGMENTS_JSONL = RUN_DIR / "judgments_multiseed_v3.jsonl"

official_tasks = load_deepnote_official_catalog(DEEPNOTE_REPO_DIR)
api_cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache_official8_v3.sqlite")
openai_client = OpenAIClient(api_key=OPENAI_API_KEY, models_cfg=MODELS_CFG, cache=api_cache, provider_name="openai")
autodl_client = (
    OpenAIClient(api_key=AUTODL_API_KEY, models_cfg=MODELS_CFG, cache=api_cache, base_url=AUTODL_BASE_URL, provider_name="autodl", timeout_sec=120.0)
    if AUTODL_API_KEY
    else None
)
triage_client = TriageClient(models_cfg=MODELS_CFG, cache=api_cache, openai_client=openai_client, luna_model="gpt-4.1-mini", jev_api_key=JEV_API_KEY)
skill_store = SkillStore.from_json_file()

multi_judge = DeepnoteOfficialJudge(
    openai_client=openai_client,
    judge_model=PRIMARY_JUDGE_MODEL,
    secondary_openai_client=autodl_client,
    secondary_judge_model=SECONDARY_JUDGE_MODEL,
    num_passes=2,
    seed=42,
)

def load_indexed_jsonl(path: Path):
    out = {}
    if not path.is_file():
        return out
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            obj = json.loads(line)
            key = (obj["method"], obj["model_id"], int(obj.get("seed", 42)), obj["task_id"])
            out[key] = obj
    return out

def append_record(path: Path, obj: dict) -> None:
    with path.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(obj, ensure_ascii=False) + "\n")
        fh.flush()
        os.fsync(fh.fileno())

completed_eps = load_indexed_jsonl(EPISODES_JSONL)
completed_judgs = load_indexed_jsonl(JUDGMENTS_JSONL)
print(f"Loaded {len(completed_eps)} cached episodes & {len(completed_judgs)} judgments | Judge={PRIMARY_JUDGE_MODEL}+{SECONDARY_JUDGE_MODEL or PRIMARY_JUDGE_MODEL}")

## Step 3 — Execute 5-Way Multi-Seed Sweep & Cross-Family Multi-Judge Grading

In [ ]:
# === CELL 5: Run 5-Way Controlled Comparison Across Backbones & Seeds ===
def build_official_methods(model_id: str, seed: int):
    collab_cfg = DialSkillAgent.build_collab_config(DIAL_CFG, enable_pre_delivery_audit=True, gate_mode="deliberative_collab")
    no_audit_cfg = DialSkillAgent.build_collab_config(DIAL_CFG, enable_pre_delivery_audit=False, gate_mode="deliberative_collab")
    return [
        (
            "B0_official_direct",
            model_id,
            OfficialDirectAgent(openai_client=openai_client, model=model_id, method_name="B0_official_direct", seed=seed),
        ),
        (
            "B0_prompt_matched_direct",
            model_id,
            OfficialPromptMatchedDirectAgent(openai_client=openai_client, model=model_id, method_name="B0_prompt_matched_direct", seed=seed),
        ),
        (
            "DIAL_v1_flat",
            f"{model_id}+{model_id}",
            OfficialDialSkillAgent(triage_client=triage_client, openai_client=openai_client, gate_model=model_id, executor_model=model_id, skill_store=skill_store, cfg=DIAL_CFG, method_name="DIAL_v1_flat", seed=seed),
        ),
        (
            "DIAL_Collab_no_stage3_audit",
            f"{model_id}+{model_id}",
            OfficialDialSkillV2Agent(triage_client=triage_client, openai_client=openai_client, gate_model=model_id, executor_model=model_id, skill_store=skill_store, cfg=no_audit_cfg, method_name="DIAL_Collab_no_stage3_audit", enable_pre_delivery_audit=False, seed=seed),
        ),
        (
            "DIAL_Collab_full",
            f"{model_id}+{model_id}",
            OfficialDialSkillV2Agent(triage_client=triage_client, openai_client=openai_client, gate_model=model_id, executor_model=model_id, skill_store=skill_store, cfg=collab_cfg, method_name="DIAL_Collab_full", enable_pre_delivery_audit=True, seed=seed),
        ),
    ]

for model_id in EVAL_MODELS:
    # Probe model availability first so unavailable preview tiers skip cleanly
    try:
        await openai_client.complete_text(model=model_id, messages=[{"role": "user", "content": "ping"}], max_tokens=5, use_cache=True)
    except Exception as exc:
        print(f"[SKIP MODEL] {model_id} unavailable on this API key: {exc}")
        continue

    for seed in SEEDS:
        print(f"\n{'=' * 84}\nOfficial-8 | Backbone={model_id} | Seed={seed}\n{'=' * 84}")
        specs = build_official_methods(model_id, seed)
        for task in official_tasks:
            for m_name, stored_mid, agent in specs:
                key = (m_name, stored_mid, seed, task.task_id)
                if key in completed_eps:
                    ep = OfficialEpisodeResult.model_validate(completed_eps[key])
                else:
                    async with DeepnoteOfficialTaskSession(
                        task=task, data_dir=OFFICIAL_DATA_DIR, cache_dir=OFFICIAL_CACHE_DIR, work_root=OFFICIAL_WORK_ROOT
                    ) as sess:
                        ep = await agent.run_task(sess)
                    ep_d = ep.model_dump(mode="json")
                    ep_d["seed"] = seed
                    append_record(EPISODES_JSONL, ep_d)
                    completed_eps[key] = ep_d

                if key in completed_judgs:
                    judg = DeepnoteRubricJudgment.model_validate(completed_judgs[key])
                else:
                    judg = await multi_judge.grade_episode(task=task, episode=ep)
                    j_d = judg.model_dump(mode="json")
                    j_d["seed"] = seed
                    append_record(JUDGMENTS_JSONL, j_d)
                    completed_judgs[key] = j_d

                print(
                    f"  [s={seed}] {task.task_id} ({task.category:10s}) | {m_name:28s} -> "
                    f"pred={ep.predicted_action:9s} writes={ep.write_calls_used:2d} reads={ep.read_calls_used:2d} score={judg.overall_score:4.1f}"
                )

## Step 4 — Multi-Seed Aggregation, Error Taxonomy, Exact Permutation Tests & LaTeX Export

In [ ]:
# === CELL 6: Aggregate Multi-Seed Mean ± Std, Paired Permutation Tests & LaTeX Table ===
import numpy as np
import pandas as pd
from dial_skill.eval.metrics import holm_bonferroni_correction, paired_nonparametric_test

rows_summary = []
per_task_mean_scores = {}

grouped_keys = sorted({(k[0], k[1]) for k in completed_eps})
for m_name, stored_mid in grouped_keys:
    seed_means = []
    seed_summaries = []
    task_acc = {t.task_id: [] for t in official_tasks}
    for seed in SEEDS:
        eps_s = [OfficialEpisodeResult.model_validate(completed_eps[(m_name, stored_mid, seed, t.task_id)]) for t in official_tasks if (m_name, stored_mid, seed, t.task_id) in completed_eps]
        j_s = {t.task_id: DeepnoteRubricJudgment.model_validate(completed_judgs[(m_name, stored_mid, seed, t.task_id)]) for t in official_tasks if (m_name, stored_mid, seed, t.task_id) in completed_judgs}
        if len(eps_s) == len(official_tasks) and len(j_s) == len(official_tasks):
            s_sum = summarize_official_8_results(eps_s, j_s)
            seed_means.append(s_sum["overall_mean"])
            seed_summaries.append(s_sum)
            for tid, j_obj in j_s.items():
                task_acc[tid].append(j_obj.overall_score)
    if not seed_summaries:
        continue
    per_task_mean_scores[(m_name, stored_mid)] = [float(np.mean(task_acc[t.task_id])) for t in official_tasks]
    last_tax = seed_summaries[0]["error_taxonomy"]
    rows_summary.append({
        "method": m_name,
        "model": stored_mid.split("+")[0],
        "n_seeds": len(seed_means),
        "overall_mean": round(float(np.mean(seed_means)), 3),
        "overall_std": round(float(np.std(seed_means, ddof=1)) if len(seed_means) > 1 else 0.0, 3),
        "silence_acc": round(float(np.mean([s["silence_accuracy"] for s in seed_summaries])), 3),
        "action_rec": round(float(np.mean([s["action_recall"] for s in seed_summaries])), 3),
        "avg_writes": round(float(np.mean([s["avg_write_calls"] for s in seed_summaries])), 2),
        "spam_eps": last_tax["multi_write_spam_episodes"],
        "serious_zeros": round(float(np.mean([s["serious_mistake_zeros"] for s in seed_summaries])), 2),
        "avg_cost_usd": round(float(np.mean([s["avg_cost_usd"] for s in seed_summaries])), 4),
    })

df_off = pd.DataFrame(rows_summary)
display(df_off)

# Exact 2^8 = 256 paired sign-flip permutation tests vs B0_official_direct & B0_prompt_matched_direct
perm_tests = {}
raw_p = {}
for model_id in EVAL_MODELS:
    ours_k = ("DIAL_Collab_full", f"{model_id}+{model_id}")
    if ours_k not in per_task_mean_scores:
        continue
    for base_m, base_mid in [
        ("B0_official_direct", model_id),
        ("B0_prompt_matched_direct", model_id),
        ("DIAL_Collab_no_stage3_audit", f"{model_id}+{model_id}"),
    ]:
        if (base_m, base_mid) in per_task_mean_scores:
            tag = f"{model_id}:DIAL_Collab_full_vs_{base_m}"
            res = paired_nonparametric_test(per_task_mean_scores[ours_k], per_task_mean_scores[(base_m, base_mid)])
            perm_tests[tag] = res
            raw_p[tag] = res["permutation_p_value"]

holm_off = holm_bonferroni_correction(raw_p)
for k, v in holm_off.items():
    perm_tests[k].update(v)

(RUN_DIR / "summary_official_8_v3_clean.json").write_text(
    json.dumps({"rows": rows_summary, "paired_tests": perm_tests}, indent=2), encoding="utf-8"
)
print("Saved clean multi-seed Official-8 summary & exact permutation tests!")